# Airbnb Multi-City Analysis — Notebook 02: Preprocessing and Schema Standardization

**Author:** Data Analyst Team  
**Project:** Airbnb Multi-City Data Analysis  
**Notebook:** 02 - Preprocessing and Schema Standardization  
**Environment:** Python + Jupyter + uv  
**Version:** 1.0  
**Last update:** 2026-04-16

---

## Business Context

After the initial data understanding phase, the project requires a formal preprocessing stage in order to transform heterogeneous raw datasets into a clean and reliable analytical base.

Because the project combines Airbnb data from multiple cities, preprocessing is especially important to ensure:
- structural consistency,
- comparable variables across markets,
- reproducible downstream analysis,
- and clean data delivery for dashboarding and modeling.

## Notebook Objective

The objective of this notebook is to:
- load all raw city-level datasets,
- harmonize their schema,
- validate and standardize data types,
- handle duplicates and missing values,
- generate a clean master dataset,
- and export the processed output for the next project stages.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Environment ready.")
print("Project running with uv-managed dependencies.")

Environment ready.
Project running with uv-managed dependencies.


## 1. Input Files and Project Paths

We define the raw data sources and the target path for the processed master dataset.

In [ ]:
data_files = {
    "Madrid": "../data/raw/madrid_airbnb.csv",
    "Milan": "../data/raw/milan_airbnb.csv",
    "London": "../data/raw/london_airbnb.csv",
    "New_York": "../data/raw/NY_airbnb.csv",
    "Sydney": "../data/raw/sydney_airbnb.csv",
    "Tokyo": "../data/raw/tokyo_airbnb.csv",
}

for city, path in data_files.items():
    if not Path(path).exists():
        raise FileNotFoundError(f"{city} file not found: {path}")

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "airbnb_clean.csv"

data_files

{'Madrid': '../data/raw/madrid_airbnb.csv',
 'Milan': '../data/raw/milan_airbnb.csv',
 'London': '../data/raw/london_airbnb.csv',
 'New_York': '../data/raw/NY_airbnb.csv',
 'Sydney': '../data/raw/sydney_airbnb.csv',
 'Tokyo': '../data/raw/tokyo_airbnb.csv'}

## 2. Load Raw Datasets

Each city dataset is loaded independently and assigned a `city` field in order to preserve source traceability after concatenation.

In [3]:
dfs = {}

for city, path in data_files.items():

    if not Path(path).exists():
        print(f"File not found: {path}")
        continue

    try:
        df = pd.read_csv(path, encoding="utf-8", low_memory=False)
    except Exception as e:
        print(f"Error loading {city}: {e}")
        continue

    if "city" in df.columns:
        print(f"Warning: 'city' already exists in {city}")

    df["city"] = city
    dfs[city] = df

print(f"Loaded datasets: {len(dfs)}")
print("Cities:", ", ".join(dfs.keys()))

Loaded datasets: 6
Cities: Madrid, Milan, London, New_York, Sydney, Tokyo


## 3. Define the Common Analytical Schema

Because the raw datasets are not fully homogeneous, the first preprocessing task is to define a common schema that can support cross-city analysis.

This schema should prioritize variables that are:
- analytically relevant,
- available in most or all cities,
- and useful for downstream EDA, hypothesis testing, modeling, and dashboarding.

In [4]:
# Core columns: required for robust cross-city analysis
# Extended columns: useful when available, but less consistent across cities

core_schema = [
    "id",
    "host_id",
    "neighbourhood",
    "latitude",
    "longitude",
    "room_type",
    "price",
    "minimum_nights",
    "number_of_reviews",
    "city"
]

extended_schema = [
    "name",
    "host_name",
    "neighbourhood_group",
    "last_review",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
]

final_schema = [
    "id",
    "name",
    "host_id",
    "host_name",
    "neighbourhood_group",
    "neighbourhood",
    "latitude",
    "longitude",
    "room_type",
    "price",
    "minimum_nights",
    "number_of_reviews",
    "last_review",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
    "city"
]

final_schema

['id',
 'name',
 'host_id',
 'host_name',
 'neighbourhood_group',
 'neighbourhood',
 'latitude',
 'longitude',
 'room_type',
 'price',
 'minimum_nights',
 'number_of_reviews',
 'last_review',
 'reviews_per_month',
 'calculated_host_listings_count',
 'availability_365',
 'city']

## 4. Schema Harmonization Strategy

The harmonization strategy is based on the following rules:

1. Keep all variables defined in the common schema.
2. If a variable is missing in a city dataset, create it with null values.
3. Reorder columns consistently across all datasets.
4. Preserve city origin for later segmentation and analysis.

In [5]:
def harmonize_schema(df: pd.DataFrame, schema: list[str], city: str = "") -> pd.DataFrame:
    df = df.copy()
    
    extra_cols = set(df.columns) - set(schema)
    if extra_cols:
        print(f"{city}: Dropping columns -> {extra_cols}")
    
    for col in schema:
        if col not in df.columns:
            df[col] = np.nan
    
    return df[schema]

In [7]:
harmonized_dfs = {}

for city, df in dfs.items():
    harmonized_dfs[city] = harmonize_schema(df, final_schema)

for city, df in harmonized_dfs.items():
    print(f"{city}: {df.shape}")

Madrid: (19618, 17)
Milan: (18322, 17)
London: (85068, 17)
New_York: (48895, 17)
Sydney: (36662, 17)
Tokyo: (11466, 17)


## 5. Concatenate the Harmonized Datasets

Once all city-level datasets have been aligned to the same schema, they can be safely concatenated into a single master DataFrame.

In [8]:
df_raw_master = pd.concat(harmonized_dfs.values(), ignore_index=True)

print("Master raw shape:", df_raw_master.shape)
display(df_raw_master.head())

Master raw shape: (220031, 17)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,city
0,6369,"Rooftop terrace room , ensuite bathroom",13660,Simon,Chamartín,Hispanoamérica,40.46,-3.68,Private room,60,1,78,2020-09-20,0.58,1.00,180.00,Madrid
1,21853,Bright and airy room,83531,Abdel,Latina,Cármenes,40.40,-3.74,Private room,31,4,33,2018-07-15,0.42,2.00,364.00,Madrid
2,23001,Apartmento Arganzuela- Madrid Rio,82175,Jesus,Arganzuela,Legazpi,40.39,-3.70,Entire home/apt,50,15,0,NaN,NaN,7.00,1.00,Madrid
3,24805,Gran Via Studio Madrid,346366726,A,Centro,Universidad,40.42,-3.71,Entire home/apt,92,5,10,2020-03-01,0.13,1.00,72.00,Madrid
4,26825,Single Room whith private Bathroom,114340,Agustina,Arganzuela,Legazpi,40.39,-3.69,Private room,26,2,149,2020-03-12,1.12,1.00,365.00,Madrid


## 6. Initial Quality Check on the Master Dataset

Now that the data has been structurally standardized, we review the master table before applying cleaning logic.

In [9]:
df_raw_master.info()

<class 'pandas.DataFrame'>
RangeIndex: 220031 entries, 0 to 220030
Data columns (total 17 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              220031 non-null  int64  
 1   name                            219964 non-null  str    
 2   host_id                         220031 non-null  int64  
 3   host_name                       219325 non-null  str    
 4   neighbourhood_group             68513 non-null   object 
 5   neighbourhood                   220031 non-null  str    
 6   latitude                        220031 non-null  float64
 7   longitude                       220031 non-null  float64
 8   room_type                       220031 non-null  str    
 9   price                           220031 non-null  int64  
 10  minimum_nights                  220031 non-null  int64  
 11  number_of_reviews               220031 non-null  int64  
 12  last_review                

In [10]:
initial_rows = len(df_raw_master)

# Duplicados exactos
duplicate_count = df_raw_master.duplicated().sum()

# Duplicados por ID
duplicate_id_count = df_raw_master["id"].duplicated().sum()

df_clean = df_raw_master.drop_duplicates().copy()

final_rows = len(df_clean)

print(f"Initial rows: {initial_rows}")
print(f"Duplicate rows removed: {duplicate_count}")
print(f"Duplicate IDs detected: {duplicate_id_count}")
print(f"Final rows after duplicate removal: {final_rows}")

Initial rows: 220031
Duplicate rows removed: 0
Duplicate IDs detected: 0
Final rows after duplicate removal: 220031


## 8. Standardize Data Types

The master dataset must use consistent data types before any analytical work is performed.

Special attention is given to:
- numerical variables,
- date fields,
- and identifier-like variables.

In [11]:
numeric_columns = [
    "id",
    "host_id",
    "latitude",
    "longitude",
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365"
]

for col in numeric_columns:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

df_clean["last_review"] = pd.to_datetime(df_clean["last_review"], errors="coerce", dayfirst=True)

C:\Users\torre\AppData\Local\Temp\ipykernel_6592\1584010224.py:18: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_clean["last_review"] = pd.to_datetime(df_clean["last_review"], errors="coerce", dayfirst=True)


In [13]:
df_clean.dtypes

id                                         int64
name                                         str
host_id                                    int64
host_name                                    str
neighbourhood_group                       object
neighbourhood                                str
latitude                                 float64
longitude                                float64
room_type                                    str
price                                      int64
minimum_nights                             int64
number_of_reviews                          int64
last_review                       datetime64[us]
reviews_per_month                        float64
calculated_host_listings_count           float64
availability_365                         float64
city                                         str
dtype: object

## 9. Missing Values Review After Harmonization

Some missing values are inherited from the original datasets, while others result from columns that were absent in certain cities and created during schema harmonization.

At this stage, the objective is to distinguish between:
- structural missingness,
- and business-level missingness.

In [14]:
missing_summary = (
    df_clean.isna()
    .sum()
    .reset_index()
)

missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df_clean) * 100).round(2)
missing_summary.sort_values(by="missing_pct", ascending=False)

,column,missing_count,missing_pct
4,neighbourhood_group,151518,68.86
12,last_review,67631,30.74
13,reviews_per_month,54371,24.71
15,availability_365,11466,5.21
14,calculated_host_listings_count,11466,5.21
3,host_name,706,0.32
1,name,67,0.03
0,id,0,0.00
2,host_id,0,0.00
8,room_type,0,0.00


## 10. Missing Value Treatment Rules

The preprocessing strategy applies context-aware rules:

- `reviews_per_month` may be set to 0 when no reviews exist.
- `last_review` may remain null when a listing has never been reviewed.
- structurally absent variables will remain null if the information was never present in the source.
- critical analytical fields should be reviewed more strictly.

In [15]:
# Rule 1: reviews_per_month = 0 when number_of_reviews = 0 and reviews_per_month is missing
condition_reviews = (
    df_clean["number_of_reviews"].fillna(0).eq(0) &
    df_clean["reviews_per_month"].isna()
)
df_clean.loc[condition_reviews, "reviews_per_month"] = 0

# Optional cleanup for host_name and name
df_clean["host_name"] = df_clean["host_name"].fillna("Unknown")
df_clean["name"] = df_clean["name"].fillna("Unknown")

## 11. Critical Field Validation

Certain variables are especially important for downstream analysis and business interpretation.

These include:
- `price`
- `room_type`
- `minimum_nights`
- `number_of_reviews`
- location-related fields
- `city`

Rows with severely incomplete core information should be reviewed carefully.

In [16]:
critical_columns = [
    "price",
    "room_type",
    "minimum_nights",
    "number_of_reviews",
    "latitude",
    "longitude",
    "city"
]

critical_missing = df_clean[critical_columns].isna().sum().sort_values(ascending=False)
critical_missing

price                0
room_type            0
minimum_nights       0
number_of_reviews    0
latitude             0
longitude            0
city                 0
dtype: int64

In [17]:
# Remove rows missing essential fields required for analysis
df_clean = df_clean.dropna(subset=["price", "room_type", "minimum_nights", "number_of_reviews", "city"]).copy()

print("Shape after dropping rows with critical missing values:", df_clean.shape)

Shape after dropping rows with critical missing values: (220031, 17)


## 12. Basic Business Rule Validation

Before proceeding, it is useful to validate some simple business assumptions:
- price should be non-negative,
- minimum_nights should be positive,
- review counts should not be negative,
- availability_365 should fall within logical bounds when available.

In [18]:
# Remove invalid values according to basic business logic
df_clean = df_clean[df_clean["price"] >= 0]
df_clean = df_clean[df_clean["minimum_nights"] > 0]
df_clean = df_clean[df_clean["number_of_reviews"] >= 0]

if "availability_365" in df_clean.columns:
    df_clean = df_clean[
        df_clean["availability_365"].isna() |
        ((df_clean["availability_365"] >= 0) & (df_clean["availability_365"] <= 365))
    ]

print("Shape after business rule validation:", df_clean.shape)

Shape after business rule validation: (220031, 17)


## 13. Standardize Categorical Text Fields

Categorical variables can contain formatting inconsistencies.  
At this stage, a light normalization is applied to support cleaner downstream analysis.

In [20]:
import numpy as np

text_columns = [
    "name",
    "host_name",
    "neighbourhood_group",
    "neighbourhood",
    "room_type",
    "city"
]

for col in text_columns:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
            .astype("string")          # asegurar tipo string
            .str.strip()               # quitar espacios
            .str.replace(r"\s+", " ", regex=True)  # espacios múltiples → uno
            .replace("", np.nan)       # strings vacíos → NaN
        )


# Normalización SOLO para categóricas
if "room_type" in df_clean.columns:
    df_clean["room_type"] = df_clean["room_type"].str.lower()

## 14. Create Minimal Business-Oriented Derived Variables

At this stage, only a minimal set of derived variables is created.

The purpose is not to engineer features for modeling yet, but to prepare simple and interpretable fields that can support:
- exploratory analysis,
- high-level business interpretation,
- and future dashboard development in Power BI.

In [21]:
# Simple derived variable useful for analysis and dashboarding
df_clean["has_reviews"] = np.where(
    df_clean["number_of_reviews"] > 0, 1, 0
)

## 15. Final Data Quality Review

After preprocessing, we perform a final validation to ensure the resulting dataset is suitable for downstream analytical use.

In [22]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 220031 entries, 0 to 220030
Data columns (total 18 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   id                              220031 non-null  int64         
 1   name                            220031 non-null  string        
 2   host_id                         220031 non-null  int64         
 3   host_name                       220031 non-null  string        
 4   neighbourhood_group             68513 non-null   string        
 5   neighbourhood                   220031 non-null  string        
 6   latitude                        220031 non-null  float64       
 7   longitude                       220031 non-null  float64       
 8   room_type                       220031 non-null  string        
 9   price                           220031 non-null  int64         
 10  minimum_nights                  220031 non-null  int64         
 11

In [23]:
final_missing_summary = (
    df_clean.isna()
    .sum()
    .reset_index()
)

final_missing_summary.columns = ["column", "missing_count"]
final_missing_summary["missing_pct"] = (final_missing_summary["missing_count"] / len(df_clean) * 100).round(2)
final_missing_summary.sort_values(by="missing_pct", ascending=False)

,column,missing_count,missing_pct
4,neighbourhood_group,151518,68.86
12,last_review,67631,30.74
14,calculated_host_listings_count,11466,5.21
15,availability_365,11466,5.21
13,reviews_per_month,123,0.06
0,id,0,0.00
1,name,0,0.00
2,host_id,0,0.00
3,host_name,0,0.00
5,neighbourhood,0,0.00


In [24]:
df_clean.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
id,"220,031.00",NaN,NaN,NaN,"22,408,312.70","2,539.00","13,383,695.00","22,497,889.00","31,554,452.00","50,955,051.00","11,754,901.81"
name,220031,213148,Unknown,67,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_id,"220,031.00",NaN,NaN,NaN,"84,945,277.80","1,944.00","14,396,024.50","46,403,919.00","141,509,678.00","411,720,762.00","88,566,073.77"
host_name,220031,31493,David,1418,NaN,NaN,NaN,NaN,NaN,NaN,NaN
neighbourhood_group,68513,26,Manhattan,21661,NaN,NaN,NaN,NaN,NaN,NaN,NaN
neighbourhood,220031,562,Westminster,9588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,"220,031.00",NaN,NaN,NaN,32.57,-34.14,40.41,40.79,51.50,51.68,30.14
longitude,"220,031.00",NaN,NaN,NaN,16.43,-74.24,-3.71,-0.13,9.20,151.34,76.03
room_type,220031,4,entire home/apt,128154,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,"220,031.00",NaN,NaN,NaN,917.82,0.00,55.00,99.00,177.00,"1,000,046.00","8,285.22"


In [25]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 220031 entries, 0 to 220030
Data columns (total 18 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   id                              220031 non-null  int64         
 1   name                            220031 non-null  string        
 2   host_id                         220031 non-null  int64         
 3   host_name                       220031 non-null  string        
 4   neighbourhood_group             68513 non-null   string        
 5   neighbourhood                   220031 non-null  string        
 6   latitude                        220031 non-null  float64       
 7   longitude                       220031 non-null  float64       
 8   room_type                       220031 non-null  string        
 9   price                           220031 non-null  int64         
 10  minimum_nights                  220031 non-null  int64         
 11

## 16. Export Processed Dataset

The cleaned and harmonized dataset is exported to the `data/processed/` directory so it can be reused consistently in:
- exploratory analysis,
- statistical testing,
- machine learning,
- and Power BI dashboard development.

In [28]:
df_clean.to_csv(output_path, index=False)

print(f"Processed dataset exported successfully to: {output_path}")

Processed dataset exported successfully to: ..\data\processed\airbnb_clean2.csv


## 17. Executive Summary

This notebook transformed the raw multi-city Airbnb data into a unified analytical dataset.

### Completed preprocessing steps
1. Loaded all city-level datasets.
2. Added source traceability through the `city` field.
3. Defined and applied a common schema.
4. Concatenated all datasets into a single master table.
5. Removed exact duplicate rows.
6. Corrected data types.
7. Applied missing value handling rules.
8. Validated critical analytical fields.
9. Applied basic business rule filtering.
10. Created derived variables for future analysis.
11. Exported the cleaned dataset for downstream use.

### Output
The project now has a clean master dataset ready for:
- exploratory data analysis,
- hypothesis testing,
- modeling,
- and dashboard construction.

## 18. Next Steps

The next notebook will focus on **Exploratory Data Analysis (EDA)** using the processed master dataset.

That stage will include:
- univariate analysis,
- price distribution analysis,
- room type and city segmentation,
- correlation analysis,
- and early business insight generation.